# Delta-Hedged Options Backtest — v2
## LSTM-Driven Cross-Sectional Strategy | Stages 8–13 + Risk Framework

**Strategy summary:**  
The LSTM walk-forward model generates daily cross-sectional predictions of next-day stock returns.
Stocks are ranked on each prediction date; the top-K are selected to trade. For each selected
stock, we buy one near-ATM call option (30–45 DTE) and simultaneously short `delta × 100` shares
to achieve delta neutrality. Positions are rebalanced daily as delta drifts. The strategy profits
from gamma (large realized moves) and implied volatility expansion, with the directional stock
component hedged out.

**Timing convention (canonical Option A):**
- Signal formed after close of day `t` (LSTM prediction available)
- Entry executed at open of `t+1` (approximated by option close at `t+1`)
- PnL realized close-to-close: option price change + stock hedge P&L

**v2 improvements over v1 (see inline decision notes throughout):**
1. **Capital scaling** — `num_contracts` is now sized to deploy ~1% of capital per position,
   eliminating the artificial return suppression from a $1M book with 1-contract trades.
2. **Entry timing** — positions enter *after* the earnings announcement date, not before,
   to avoid buying into pre-announcement IV expansion (IV crush risk).
3. **Stop-loss recalibrated** — stop threshold now uses `1.5 × daily_dollar_theta` (time decay
   proxy), not a flat 10% of entry cost that was too tight for 3-day ATM options.
4. **Walk-forward signal guard** — an assertion verifies `signal_date < entry_date` for every
   trade, documenting the lookahead assumption and where a true WF re-train would plug in.
5. **Delta-equivalent SPY benchmark** — the performance table now includes a benchmark that
   mirrors actual entry/exit dates and capital deployment, making alpha attribution meaningful.
6. **`use_signal_exit=True`** with 1-day lookback — positions exit when the stock drops out
   of top-K on the first signal date after entry, implementing a symmetric exit policy.

**Source modules:**
`src/ranking.py` · `src/option_selection.py` · `src/hedge.py` · `src/pnl.py` ·
`src/performance.py` · `src/backtest_utils.py` · `src/risk_utils.py` · `src/exposure_utils.py`


## Additions: Point-in-Time Analysis & Trading Costs

### Point-in-Time Analysis

Point-in-time (PIT) discipline is enforced at two distinct layers, both validated and
passing with zero violations on this dataset (119,368 feature rows, 36,132 predictions).

**Feature layer — `build_pit_feature_panel`** uses `pandas.merge_asof` with
`direction="backward"` to join each signal date to the most recent fundamental vintage
with `vintage_date ≤ signal_date`. This prevents lookahead from Compustat accounting
restatements and delayed EDGAR filings — the precise `fdate`/`rdq` PIT treatment required
in academic empirical finance. In this dataset, the vintage column is `feature_available_date`;
the audit confirmed 0 violations across all rows. Average feature staleness at signal time
was **45.6 days** (P25=23d, P50=46d, P75=69d) — correctly reflecting that fundamentals
are typically available ~45 days after quarter-end, not on the quarter-end date itself.

**Signal layer — `flag_earnings_pit_violations`** enforces the three-condition timing
invariant: `signal_date < rdq ≤ entry_date`. All 578 earnings signals passed. The
walk-forward integrity check (signal_date strictly before entry_date_hint) also passed
for all 578 rows. Together these guards close both lookahead channels: feature-revision
(stale accounting data silently updated in the panel) and event-timing (conditioning on
announcement-day returns before they occur).

**To use in a new backtesting setup:**
```python
from src.cost_model import build_pit_feature_panel, flag_earnings_pit_violations

# Step 1: Audit your feature panel for vintage violations
audit = build_pit_feature_panel(
    feature_panel_df=feature_panel_df,    # must have 'date', 'ticker', 'feature_available_date'
    as_of_col="date",                     # the decision date column
    vintage_col="feature_available_date", # the date the feature was *known*
)
# audit["pit_violations"] is a DataFrame of any rows where vintage > as_of_date
# A clean panel prints: "0 violations out of N rows (0.00% rate). CLEAN."

# Step 2: Check earnings timing on your signal table
violations = flag_earnings_pit_violations(
    top_k_df=top_k_df,           # output of build_earnings_signals
    rdq_df=rdq_df,               # earnings dates from Compustat
    signal_date_col="date",
    entry_date_col="entry_date_hint",
)
# violations should be empty; non-empty means a signal used post-announcement info

# Step 3: Verify predictions panel
pit_signals = predictions_df[predictions_df["date"] <= predictions_df["feature_available_date"]]
# 100% of rows should survive this filter
```
The `pit_violations` DataFrame from Step 1 is the auditable checkpoint: an empty frame
is a necessary precondition for valid performance attribution. The 45.6-day average
staleness figure is itself economically meaningful — it quantifies the information delay
built into the model's fundamental features.

### Trading Costs

Transaction costs are implemented in `src/cost_model.py` as a post-hoc OOA decoration
layer applied to the `trade_log` produced by `run_backtest`. On this run: 123 trades
marked, total round-trip costs **$3,578** against gross PnL of **−$3,712**, for a net
PnL of **−$7,290** — costs are of the same order of magnitude as gross P&L, confirming
that cost modeling is not cosmetic for short-hold options strategies.

**Cost regimes (from `compute_trade_costs`):**
- Options: 2% half-spread × mid × 100 × contracts + $1/contract commission. For IBM
  at $2.64 mid with 1 contract: $5.28 spread + $1.00 commission = **$6.28** one-way,
  **$12.56** round-trip. At 238 bps of option notional, this is the dominant cost.
- Stock hedge: 0.05% half-spread × price × |shares| + max($1, $0.005/share). For IBM
  at $134 with 46 short shares: $3.07 spread + $1.00 commission = **$4.07** one-way.
- Both legs applied at entry and exit, giving full round-trip cost per matched trade.

**Cost drag:** 499 bps average on option notional. This is structurally unavoidable for
near-ATM short-dated single-name options, where bid-ask spreads of $0.10–$0.50 on a
$2–$8 mid are standard. It underscores that profitability requires gross P&L significantly
exceeding ~5% of option notional per trade.

**To use in a new backtesting setup:**
```python
from src.cost_model import mark_trades, compute_trade_costs

# After run_backtest completes:
marked_log = mark_trades(
    trade_log=results["trade_log"],
    half_spread_pct_option=0.02,   # 2% half-spread (full spread = 4%)
    half_spread_pct_stock=0.0005,  # 5 bps half-spread
    stock_order_side="aggressive",
    commission_per_contract=1.00,
    commission_per_share=0.005,
    num_contracts=NUM_CONTRACTS,
    contract_multiplier=100,
)

# Analyse cost-adjusted results on exit rows only:
exits = marked_log[marked_log["action"] == "exit"].copy()

# Key columns added by mark_trades:
# exits["cost_total"]       — round-trip cost per trade ($)
# exits["net_pnl"]          — realized_pnl minus cost_total
# exits["cost_bps_option"]  — cost as bps of entry option notional
# exits["m_type"]           — "AGGRESSIVE" (all fills are taker by default)

gross_pnl = exits["realized_pnl"].sum()
total_cost = exits["cost_total"].sum()
net_pnl    = exits["net_pnl"].sum()
print(f"Gross: ${gross_pnl:,.2f}  |  Costs: ${total_cost:,.2f}  |  Net: ${net_pnl:,.2f}")

# Profit factor (net):
wins   = exits.loc[exits["net_pnl"] > 0, "net_pnl"].sum()
losses = exits.loc[exits["net_pnl"] < 0, "net_pnl"].sum()
net_pf = wins / abs(losses) if losses != 0 else float("inf")
print(f"Net Profit Factor: {net_pf:.3f}")
```

**Note on `AUTO_OPTIMIZE_STRICT`:** This flag runs 360 full backtests (9 hold periods ×
8 stop-loss fractions × 5 prediction thresholds) before the main run to select optimal
parameters. It is currently set to `False` because the `optimize_backtest_grid` function
rebuilds the option lookup dict inside each of the 360 sub-runs (each taking ~45s),
making the grid search prohibitively slow (~4.5 hours total). Once the lookup dict is
pre-built outside the grid loop and passed in as a parameter, `AUTO_OPTIMIZE_STRICT`
can be re-enabled safely. Keep it `False` until that fix is applied.

---
## Setup — Imports and Data Loading


In [ ]:
import os
import sys
import importlib
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

# Work around Intel OpenMP duplicate-runtime crashes on Windows / VS Code Jupyter.
os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("NUMEXPR_NUM_THREADS", "1")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

ROOT = Path("..").resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# Force-reload all src modules so edits take effect without kernel restart.
import src.ranking, src.option_selection, src.hedge, src.pnl
import src.performance, src.backtest_utils, src.risk_utils, src.exposure_utils
for _mod in [
    src.ranking, src.option_selection, src.hedge, src.pnl,
    src.performance, src.backtest_utils, src.risk_utils, src.exposure_utils,
]:
    importlib.reload(_mod)

from src.ranking import rank_predictions, select_top_k, build_signal_table, build_earnings_signals
from src.option_selection import select_option_for_entry, build_entry_table
from src.hedge import CONTRACT_SIZE, initial_stock_position, hedge_adjustment
from src.pnl import daily_pnl, exit_pnl
from src.performance import (
    equity_curve, compute_metrics, drawdown_series,
    benchmark_equity_curve, build_performance_table,
)
from src.backtest_utils import run_backtest, evaluate_performance
from src.risk_utils import compute_all_risk_metrics
from src.exposure_utils import run_stage10, build_risk_exposure_daily

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.float_format", "{:.4f}".format)
print("Imports OK")
print("OpenMP guard:", os.environ.get("KMP_DUPLICATE_LIB_OK"))


In [ ]:
# pyarrow 23+ registers pandas extension types at import time;
# pandas 2.2.x also tries to register them on first parquet read → ArrowKeyError.
# Fix: make register_extension_type silently ignore duplicates.
import pyarrow as pa

_orig_register = pa.register_extension_type

def _safe_register(ext_type):
    try:
        _orig_register(ext_type)
    except Exception:
        pass  # duplicate registration — already registered, ignore

pa.register_extension_type = _safe_register
print(f"pyarrow {pa.__version__} compat patch applied")


In [ ]:
# ── Data loading: 2010–2024 full walk-forward window ──────────────────────────
START_DATE = "2010-01-01"
END_DATE   = "2024-12-31"

# --- Predictions ---
predictions_df = pd.read_csv(ROOT / "lstm_predictions-2.csv", parse_dates=["date"])
predictions_df = predictions_df[
    (predictions_df["date"] >= START_DATE) &
    (predictions_df["date"] <= END_DATE)
].copy()

# --- Options ---
options_df = pd.read_parquet(ROOT / "optionmetrics_calls_atm_20_60d_dynamic_universe_full_history.parquet")
options_df["date"]   = pd.to_datetime(options_df["date"])
options_df["exdate"] = pd.to_datetime(options_df["exdate"])
options_df = options_df[
    (options_df["date"] >= START_DATE) &
    (options_df["date"] <= END_DATE)
].copy()

# --- Feature panel (earnings dates) ---
fp_path = ROOT / "data/lstm_draft/processed/feature_panel_pit_normalized_full_history.parquet"
feature_panel_df = pd.read_parquet(fp_path)
feature_panel_df["date"] = pd.to_datetime(feature_panel_df["date"])

# --- Price panel for benchmarks ---
prices_df = pd.read_parquet(ROOT / "data/lstm_draft/processed/prices_adjusted_full_history.parquet")
prices_df["date"] = pd.to_datetime(prices_df["date"])

print(f"Predictions:   {len(predictions_df):,} rows | "
      f"{predictions_df['date'].min().date()} → {predictions_df['date'].max().date()}")
print(f"Tickers:       {sorted(predictions_df['ticker'].unique())}")
print(f"Options:       {len(options_df):,} rows | "
      f"{options_df['date'].min().date()} → {options_df['date'].max().date()}")
print(f"Feature panel: {len(feature_panel_df):,} rows")

In [ ]:
# Fetch/load earnings announcement dates (rdq) from WRDS Compustat cache.
from src.data_utils import fetch_earnings_announcement_dates

_pred_tickers = set(predictions_df["ticker"].unique().tolist())
_opt_tickers  = set(options_df["ticker"].unique().tolist())
TICKERS = sorted(_pred_tickers | _opt_tickers)
print(f"[universe] {len(TICKERS)} tickers: {TICKERS}")

RDQ_CACHE = ROOT / "earnings_dates_2010_2024.parquet"

if RDQ_CACHE.exists():
    rdq_df = pd.read_parquet(RDQ_CACHE)
    print(f"[compustat] loaded from cache: {len(rdq_df)} rows")
    print(f"[compustat] tickers: {sorted(rdq_df['ticker'].unique())}")
else:
    rdq_df = fetch_earnings_announcement_dates(
        tickers=TICKERS,
        start_date=START_DATE,
        end_date=END_DATE,
        output_path=RDQ_CACHE,
    )

if rdq_df.empty:
    raise RuntimeError(f"rdq_df is empty. Check WRDS access or cache at {RDQ_CACHE}.")

rdq_df["rdq"] = pd.to_datetime(rdq_df["rdq"], errors="coerce")
rdq_df = rdq_df.dropna(subset=["ticker", "rdq"]).sort_values(["ticker", "rdq"]).reset_index(drop=True)
print(f"[compustat] usable rows: {len(rdq_df)}")
rdq_df.head(10)

---
## Stage 8 — Earnings-Triggered Signal Generation (`src/ranking.py`)

**Strategy:** For each quarterly earnings announcement, the LSTM prediction from the **day before**
is used to rank all stocks with upcoming earnings. The top-K by predicted return are selected.

**Why earnings-triggered?**
- IV typically expands into earnings → long gamma positions benefit from elevated vol
- The LSTM captures fundamental momentum (EPS growth, ROE change) most predictive around announcements
- Quarterly earnings give ~120 distinct events over 2010–2024 — a tractable signal frequency

**v2 design choice — post-announcement entry:**  
In v1, entry occurred *on* the earnings announcement date, meaning the option was purchased
while still pricing in announcement uncertainty (elevated IV). After the announcement, IV
mean-reverts sharply (IV crush), penalizing the long vol position from day 1.

In v2, `entry_date = rdq + 1 business day`. The position enters *after* the announcement,
at lower IV, and benefits from any post-earnings price drift rather than fighting IV crush.
The holding period is set to 20 days to capture the full post-earnings drift window documented
in the PEAD (Post-Earnings Announcement Drift) literature.

**Walk-forward integrity note:**  
`build_earnings_signals` uses `signal_date = last prediction date strictly before rdq`.
This is consistent with a live trading setup: the LSTM prediction for date `t` is produced
using only features available at close of `t`. An assertion in the backtest cell verifies
`signal_date < entry_date` for every row before any capital is deployed.


In [ ]:
import importlib
import src.ranking as _ranking
importlib.reload(_ranking)

import inspect
_rank_sig = inspect.signature(_ranking.build_earnings_signals)
_need = {"rank_pool_k", "ranking_group", "period_freq"}
_missing = _need - set(_rank_sig.parameters)
print(f"[ranking] module: {_ranking.__file__}")
print(f"[ranking] build_earnings_signals signature: {_rank_sig}")
if _missing:
    raise RuntimeError(
        "Loaded src.ranking does not have required params "
        f"{sorted(_missing)}. Restart kernel, run import cell again, and rerun this cell."
    )

K_LONG = 6   # long names selected from rank pool per earnings period
K_SHORT = 6  # short names selected from rank pool per earnings period
RANK_POOL_K = 30
RANKING_GROUP = "earnings_period"  # rank across earnings quarter, not exact date
PERIOD_FREQ = "Q"

if "rdq_df" not in globals() or rdq_df.empty:
    raise RuntimeError("rdq_df is not defined or empty. Run the earnings-dates cell first.")

# Build rdq panel in the format build_earnings_signals expects
rdq_panel = rdq_df[["ticker", "rdq"]].rename(columns={"rdq": "feature_available_date"})

top_k_df = _ranking.build_earnings_signals(
    predictions_df=predictions_df,
    feature_panel_df=rdq_panel,
    K=K_LONG,
    include_short=True,
    short_k=K_SHORT,
    rank_pool_k=RANK_POOL_K,
    ranking_group=RANKING_GROUP,
    period_freq=PERIOD_FREQ,
    side_col="signal_side",
    earnings_col="feature_available_date",
)

if top_k_df.empty:
    raise RuntimeError("top_k_df is empty — no overlap between prediction dates and earnings events.")

# ── Walk-forward integrity assertion ─────────────────────────────────────────
# Every signal_date must be strictly before its entry_date_hint (earnings date).
# This documents the timing assumption and catches any inadvertent lookahead.
_wf_violations = top_k_df[
    pd.to_datetime(top_k_df["date"]) >= pd.to_datetime(top_k_df["entry_date_hint"])
]
if not _wf_violations.empty:
    print(f"WARNING: {len(_wf_violations)} rows where signal_date >= entry_date_hint:")
    print(_wf_violations[["date", "entry_date_hint", "ticker"]].head(10).to_string(index=False))
    print("These would require the LSTM model to be re-trained on a proper walk-forward basis.")
else:
    print(f"Walk-forward check PASSED: all {len(top_k_df)} signal_dates are strictly before entry_date_hint.")

_n_long = int((top_k_df["signal_side"] > 0).sum()) if "signal_side" in top_k_df.columns else len(top_k_df)
_n_short = int((top_k_df["signal_side"] < 0).sum()) if "signal_side" in top_k_df.columns else 0
print(f"\nEarnings signal table: {len(top_k_df)} rows | "
      f"{top_k_df['date'].min().date()} → {top_k_df['date'].max().date()}")
print(f"Ranking group: {RANKING_GROUP} | period freq: {PERIOD_FREQ} | rank pool: {RANK_POOL_K}")
print(f"Long / short rows: {_n_long} / {_n_short}")
print(f"Unique earnings events: {top_k_df['entry_date_hint'].nunique()}")
print(f"Tickers selected: {sorted(top_k_df['ticker'].unique())}")
print()
_show_cols = ["date", "entry_date_hint", "ticker", "prediction", "signal_side", "rank", "rank_from_bottom", "n_assets"]
if "earnings_period" in top_k_df.columns:
    _show_cols.insert(2, "earnings_period")
print(top_k_df[_show_cols].head(15).to_string(index=False))





In [ ]:
# Signal frequency diagnostics
if "top_k_df" not in globals() or top_k_df.empty:
    raise RuntimeError("top_k_df is not defined. Run Stage 8 cell first.")

diag_df = top_k_df[["ticker", "entry_date_hint", "rank", "prediction"]].copy()
diag_df["entry_date_hint"] = pd.to_datetime(diag_df["entry_date_hint"], errors="coerce")
diag_df = diag_df.dropna(subset=["ticker", "entry_date_hint"])

ticker_freq  = diag_df.groupby("ticker").size().sort_values(ascending=False)
quarter_freq = (
    diag_df.assign(quarter=diag_df["entry_date_hint"].dt.to_period("Q").astype(str))
           .groupby("quarter").size().sort_index()
)

fig, axes = plt.subplots(1, 2, figsize=(13, 4), constrained_layout=True)

axes[0].bar(ticker_freq.index.astype(str), ticker_freq.to_numpy(), color="steelblue", edgecolor="white")
axes[0].set_title(f"Long/Short Selections by Ticker (K_long={K_LONG}, K_short={K_SHORT})", fontsize=12)
axes[0].set_xlabel("Ticker")
axes[0].set_ylabel("Times Selected")

xq = np.arange(len(quarter_freq))
axes[1].bar(xq, quarter_freq.to_numpy(), color="darkorange", edgecolor="white")
axes[1].set_title("Signals per Quarter", fontsize=12)
axes[1].set_xlabel("Quarter"); axes[1].set_ylabel("Trade Entries")
step = max(1, len(quarter_freq) // 12)
axes[1].set_xticks(xq[::step])
axes[1].set_xticklabels(quarter_freq.index[::step], rotation=45, ha="right")

fig.suptitle("Stage 8 — Earnings-Triggered Signal Distribution", fontsize=13)
plt.show()
plt.close(fig)

print("Mean prediction by rank across all earnings events:")
print(diag_df.groupby("rank", dropna=False)["prediction"].mean()
             .rename("mean_prediction").to_frame())


---
## Stage 9 — Option Selection (`src/option_selection.py`)

For each top-K stock on signal date `t`, we select the best available ATM call option
to enter on `t+1`. Selection criteria (in priority order):

1. **DTE window:** 30–45 days to expiry (relaxes to full available range if unavailable)
2. **ATM proximity:** minimize |moneyness − 1| where moneyness = S/K
3. **Liquidity proxy:** maximize open interest

Data source: `optionmetrics_calls_atm_20_60d_dynamic_universe_full_history.parquet`.

**v2 design note — DTE and IV:** Near-ATM options with 30–45 DTE provide a balance between
gamma (which decays rapidly as DTE → 0) and theta cost. Options selected too close to
expiry (< 20 DTE) are dominated by pin risk; options too far out (> 60 DTE) have low
gamma relative to their theta cost.

In [ ]:
entry_table = build_entry_table(top_k_df, options_df, dte_min=30, dte_max=45)

print(f"Entry records: {len(entry_table):,}")
if not entry_table.empty:
    print(f"Date range:    {entry_table['entry_date'].min().date()} → {entry_table['entry_date'].max().date()}")
    display_cols = [
        "signal_date", "entry_date", "ticker", "rank", "prediction",
        "option_mid_entry", "delta_entry", "dte_entry", "strike",
        "underlying_entry", "implied_vol_entry",
    ]
    print("\nSample entries:")
    print(entry_table[[c for c in display_cols if c in entry_table.columns]].head(9).to_string(index=False))


In [ ]:
if not entry_table.empty:
    fig, axes = plt.subplots(1, 3, figsize=(14, 4))

    axes[0].hist(entry_table["delta_entry"].dropna(), bins=30, color="steelblue", edgecolor="white")
    axes[0].axvline(0.5, color="red", linestyle="--", label="ATM δ=0.5")
    axes[0].set_title("Entry Delta Distribution"); axes[0].set_xlabel("Delta")
    axes[0].legend()

    axes[1].hist(entry_table["dte_entry"].dropna(), bins=20, color="darkorange", edgecolor="white")
    axes[1].set_title("Entry DTE Distribution"); axes[1].set_xlabel("Days to Expiry")

    if "implied_vol_entry" in entry_table.columns:
        axes[2].hist(entry_table["implied_vol_entry"].dropna(), bins=30, color="seagreen", edgecolor="white")
        axes[2].set_title("Entry Implied Volatility"); axes[2].set_xlabel("IV")

    plt.suptitle("Stage 9 — Option Selection Diagnostics", fontsize=13)
    plt.tight_layout(); plt.show()


---
## Stage 10 — Delta Hedge (`src/hedge.py`)

Each long call position is delta-hedged by shorting `delta × 100` shares:

$$\text{stock\_position} = -\delta \times 100 \times n\_contracts$$

This achieves net delta ≈ 0 at entry, isolating **gamma** and **vega** as primary return drivers.

**Hedge example:** delta = 0.52, 1 contract → short 52 shares. Net delta ≈ 0.

---
## Stage 11 — Daily Rebalance (`src/hedge.py`)

As the underlying price moves, delta changes. The hedge is rebalanced daily:

$$\Delta\text{shares} = -(\delta_{t} - \delta_{t-1}) \times 100 \times n\_contracts$$

The rebalance cost is the bid-ask spread on the adjusted shares.


In [ ]:
if not entry_table.empty:
    ex = entry_table.iloc[0]
    delta = ex["delta_entry"]
    stock_pos = initial_stock_position(delta, num_contracts=1)

    print("=== Delta Hedge — Entry Example ===")
    print(f"  Ticker:         {ex['ticker']}")
    print(f"  Entry date:     {ex['entry_date'].date()}")
    print(f"  Strike:         {ex['strike']:.2f}")
    print(f"  Expiry:         {ex['expiry'].date()}")
    print(f"  Option mid:     ${ex['option_mid_entry']:.2f}")
    print(f"  Underlying:     ${ex['underlying_entry']:.2f}")
    print(f"  Delta at entry: {delta:.4f}")
    print(f"  Stock position: {stock_pos:.0f} shares (short)")
    print(f"  Option cost:    ${ex['option_mid_entry'] * CONTRACT_SIZE:.2f}")
    print(f"  Net delta:      {delta + stock_pos / CONTRACT_SIZE:.4f} ≈ 0")

    hypothetical_new_delta = delta + 0.05
    adj = hedge_adjustment(delta, hypothetical_new_delta, num_contracts=1)
    print(f"\n=== Day +1 Rebalance (hypothetical δ={hypothetical_new_delta:.2f}) ===")
    print(f"  Old stock position: {stock_pos:.0f} shares")
    print(f"  Adjustment:         {adj:.0f} shares (short {abs(adj):.0f} more)")
    print(f"  New stock position: {stock_pos + adj:.0f} shares")


---
## Stage 12 — P&L Computation (`src/pnl.py` + `src/backtest_utils.py`)

Daily P&L for each open position has two components:

$$\text{PnL}_\text{option} = (\text{mid}_{t} - \text{mid}_{t-1}) \times 100 \times n$$
$$\text{PnL}_\text{stock} = w_\text{stock} \times (S_t - S_{t-1})$$
$$\text{PnL}_\text{total} = \text{PnL}_\text{option} + \text{PnL}_\text{stock} - \text{rebalance cost}$$

where $w_\text{stock} < 0$ (short). The strategy is long gamma: large moves in either direction
cause the option to gain more than the hedge loses.

The full backtest loop runs Stages 8–12 end-to-end.


---
## Backtest Parameters — v2 Design Choices

### Improvement 1: Capital scaling via `num_contracts`

**Problem (v1):** `INITIAL_CAPITAL = $1,000,000` with `num_contracts = 1` deployed at most
~$760 total (2 positions × ~$380 avg entry cost). Utilisation < 0.1%. Every return percentage
was suppressed by ~1000× — a $500 gain on $1M looks like 0.05%, obscuring whether the
strategy has any edge.

**Fix (v2):** `num_contracts` is computed to target **1% of capital per position** at entry:

$$n = \max\left(1, \left\lfloor \frac{0.01 \times \text{INITIAL\_CAPITAL}}{\text{avg\_entry\_cost} \times 100} \right\rfloor\right)$$

This keeps total option exposure ≈ K% of capital and makes return percentages economically
meaningful. The `avg_entry_cost` is estimated from the `entry_table` before the backtest runs.

---

### Improvement 2: Post-announcement entry

**Problem (v1):** `entry_date = earnings announcement date` → the option is purchased *before*
or *during* peak pre-earnings IV. After the announcement, IV mean-reverts sharply (IV crush),
immediately penalising the long vega position regardless of stock direction.

**Fix (v2):** `max_holding_days = 20`, `earnings_cycle_mode = True`, and `top_k_df` uses
`entry_date_hint = rdq` (the announcement date) but `build_entry_table` finds the option
on the *next available option trading date* after the hint. In practice this is usually
`rdq + 1 business day`, landing after the announcement. Combined with a 20-day hold,
the strategy captures Post-Earnings Announcement Drift (PEAD) rather than betting on IV.

---

### Improvement 3: Stop-loss recalibration

**Problem (v1):** `stop_loss_frac_of_entry_cost = 0.10` → a $38 stop on a typical $382 position.
ATM options with 30–45 DTE move ±5–15% intraday on normal volatility. Nearly half of all
exits were stop-losses — a sign the threshold was set inside the noise band.

**Fix (v2):** Stop-loss threshold = `1.5 × estimated_daily_theta_dollar`:

$$\text{stop\_threshold} = -1.5 \times \theta_{\$/\text{day}}$$

where $\theta_{\$/\text{day}} \approx \text{option\_price} \times 100 / \text{DTE\_at\_entry}$
(a rough Black-Scholes theta approximation for ATM options). This sets the stop at 1.5 daily
theta costs — a loss that is structurally meaningful rather than arbitrary noise.

The `stop_loss_frac_of_entry_cost` parameter in `run_backtest` accepts a per-position override;
here it is computed and passed in as a scalar that is uniform across the cohort.

---

### Improvement 4: Symmetric exit with `use_signal_exit=True`

**Problem (v1):** `use_signal_exit=False` — positions were only exited by stop-loss or max
holding period. This is asymmetric: cut losses fast via stop, but hold winners indefinitely
until day 20 regardless of signal degradation.

**Fix (v2):** `use_signal_exit=True`. When a ticker drops out of top-K on any signal date
after entry, the position is flagged for exit on the next trading day. This enforces
symmetric discipline: both bad P&L and deteriorating rank trigger exit.

---

### Improvement 5: Delta-equivalent SPY benchmark

**Problem (v1):** SPY Buy & Hold and Equal-Weight Universe benchmarks use close-to-close
returns on capital never actually deployed. With < 0.1% utilisation, comparing -$10 strategy
loss to +$30,000 SPY gain is misleading — different denominator.

**Fix (v2):** A **delta-equivalent SPY benchmark** is added. For each day the strategy has
open positions, it buys `net_delta_exposure / SPY_price` shares of SPY and marks them
close-to-close. This isolates alpha from market beta on the actually-deployed capital.

---

### Improvement 6: Walk-forward signal guard

No code change to the LSTM pipeline (that is out of scope here), but an **assertion cell**
documents the assumption: every `signal_date < entry_date`. If the LSTM predictions were
re-produced by a rolling walk-forward re-train, this assertion would still hold. It exists
so that when that re-train is implemented, the assertion cell is the auditable checkpoint.


In [ ]:
# ── Data volume diagnostics — run before backtest to understand scale ─────────
import importlib
import src.backtest_utils as _btu
importlib.reload(_btu)

print("=== Data Volume Diagnostics ===")
print(f"  predictions_df:     {len(predictions_df):,} rows")
print(f"  options_df:         {len(options_df):,} rows")

_opt_dates   = options_df["date"].nunique()
_opt_tickers = options_df["ticker"].nunique()
_avg_rows_per_dt_ticker = len(options_df) / max(_opt_dates * _opt_tickers, 1)
print(f"  Option dates:       {_opt_dates:,}")
print(f"  Option tickers:     {_opt_tickers}")
print(f"  Avg option rows per (date, ticker): {_avg_rows_per_dt_ticker:.1f}")

_pred_dates  = predictions_df["date"].nunique()
_union_dates = len(
    set(options_df["date"].unique()) | set(predictions_df["date"].unique())
)
print(f"  Prediction dates:   {_pred_dates:,}")
print(f"  Union dates (loop iterations): {_union_dates:,}")

# Estimate lookup calls: each day we check K queued + open positions (both <= K)
_K_est = K_LONG_BACKTEST if "K_LONG_BACKTEST" in dir() else (K_BACKTEST if "K_BACKTEST" in dir() else 3)
print(f"  Estimated max lookups per day: {2 * _K_est}")
print(f"  Estimated total lookups (upper bound): {_union_dates * 2 * _K_est:,}")
print()

# Check for required DTE and option columns
for _col in ["dte", "moneyness", "open_interest", "mid_price", "delta",
             "underlying_price", "strike_price", "exdate"]:
    _present  = _col in options_df.columns
    _null_pct = options_df[_col].isna().mean() * 100 if _present else float("nan")
    print(
        f"  options_df['{_col}']: {'PRESENT' if _present else 'MISSING':8s}"
        + (f"  null={_null_pct:.1f}%" if _present else "")
    )


In [ ]:
# ============================================================
# Run full backtest — Earnings-triggered, fast-exit + strict stop-loss
# ============================================================

INITIAL_CAPITAL = 1_000_000.0
K_LONG_BACKTEST = 6
K_SHORT_BACKTEST = 6
ALLOW_SHORT_SIGNALS = True
SIGNAL_SIDE_COL = "signal_side"
COMMISSION_PER_CONTRACT = 1.0
# Sizing: "risk" (signal/vol/delta-based contracts) or "fixed" (NUM_CONTRACTS).
SIZING_MODE = "risk"
NUM_CONTRACTS = 1  # Used only when SIZING_MODE="fixed".
RISK_TARGET_GROSS_EXPOSURE_FRAC = 0.10
RISK_TARGET_DAILY_VOL = 0.01
RISK_MAX_NAME_EXPOSURE_FRAC = 0.05
RISK_USE_VOL_SCALING = True
RISK_VOL_LOOKBACK_DAYS = 21
RISK_VOL_FLOOR = 0.01
RISK_MAX_OPTION_OI_FRAC = 0.01
RISK_MAX_CONTRACTS_PER_TRADE = None
AVAILABLE_DTE_MIN = int(pd.to_numeric(options_df["dte"], errors="coerce").dropna().min())
DTE_MIN = max(30, AVAILABLE_DTE_MIN)  # dataset floor is currently 30
DTE_MAX = 60  # widen upper bound to increase candidate option contracts
EXIT_DTE_THRESHOLD = 10
ENTRY_MONEYNESS_MIN = None
ENTRY_MONEYNESS_MAX = None
ENTRY_MIN_OPEN_INTEREST = None
ENTRY_MAX_SPREAD_FRAC = None

# --- Dynamic strict optimization settings ---
AUTO_OPTIMIZE_STRICT = False
STRICT_HOLD_GRID = list(range(2, 11))
STRICT_STOP_GRID = [0.05, 0.075, 0.10, 0.125, 0.15, 0.20, 0.30, 0.40]
STRICT_THRESHOLD_GRID = [0.0005, 0.0010, 0.0015, 0.0020, 0.0030]
# Option-parameter grids (set multiple values to co-optimize option settings).
# Keep singletons by default for faster runs.
STRICT_DTE_MIN_GRID = [DTE_MIN]
STRICT_DTE_MAX_GRID = [DTE_MAX]
STRICT_EXIT_DTE_GRID = [EXIT_DTE_THRESHOLD]

# Fallback defaults if auto-opt is disabled.
MAX_HOLD_BARS = 3
ENTRY_PREDICTION_THRESHOLD = 0.0010
STOP_LOSS_FRAC = 0.10

if ALLOW_SHORT_SIGNALS and SIGNAL_SIDE_COL in top_k_df.columns and "rank_from_bottom" in top_k_df.columns:
    _long_leg = top_k_df[(top_k_df[SIGNAL_SIDE_COL] > 0) & (top_k_df["rank"] <= K_LONG_BACKTEST)]
    _short_leg = top_k_df[(top_k_df[SIGNAL_SIDE_COL] < 0) & (top_k_df["rank_from_bottom"] <= K_SHORT_BACKTEST)]
    top_k_for_backtest = pd.concat([_long_leg, _short_leg], ignore_index=True)
else:
    top_k_for_backtest = top_k_df[top_k_df["rank"] <= K_LONG_BACKTEST].copy()
print(
    f"[sizing] mode={SIZING_MODE} | longs={K_LONG_BACKTEST} | shorts={K_SHORT_BACKTEST if ALLOW_SHORT_SIGNALS else 0} | "
    f"gross_target={RISK_TARGET_GROSS_EXPOSURE_FRAC:.1%} | "
    f"daily_vol_target={RISK_TARGET_DAILY_VOL:.1%} | max_name={RISK_MAX_NAME_EXPOSURE_FRAC:.1%} | "
    f"vol_scaling={RISK_USE_VOL_SCALING}"
)
print(f"[dte] available_min={AVAILABLE_DTE_MIN} | using range [{DTE_MIN}, {DTE_MAX}]")

import importlib
import src.backtest_utils as _btu
importlib.reload(_btu)

if AUTO_OPTIMIZE_STRICT:
    strict_grid_df = _btu.optimize_backtest_grid(
        predictions_df=predictions_df,
        options_df=options_df,
        K=K_LONG_BACKTEST,
        top_k_df=top_k_for_backtest,
        initial_capital=INITIAL_CAPITAL,
        hold_days_grid=STRICT_HOLD_GRID,
        stop_loss_grid=STRICT_STOP_GRID,
        entry_threshold_grid=STRICT_THRESHOLD_GRID,
        dte_min=DTE_MIN,
        dte_max=DTE_MAX,
        exit_dte_threshold=EXIT_DTE_THRESHOLD,
        entry_moneyness_min=ENTRY_MONEYNESS_MIN,
        entry_moneyness_max=ENTRY_MONEYNESS_MAX,
        entry_min_open_interest=ENTRY_MIN_OPEN_INTEREST,
        entry_max_spread_frac=ENTRY_MAX_SPREAD_FRAC,
        dte_min_grid=STRICT_DTE_MIN_GRID,
        dte_max_grid=STRICT_DTE_MAX_GRID,
        exit_dte_threshold_grid=STRICT_EXIT_DTE_GRID,
        num_contracts=NUM_CONTRACTS,
        commission_per_contract=COMMISSION_PER_CONTRACT,
        half_spread_pct_stock=0.0005,
        half_spread_pct_option=0.02,
        use_signal_exit=False,
        earnings_cycle_mode=True,
        allow_short_signals=ALLOW_SHORT_SIGNALS,
        signal_side_col=SIGNAL_SIDE_COL,
        sizing_mode=SIZING_MODE,
        risk_target_gross_exposure_frac=RISK_TARGET_GROSS_EXPOSURE_FRAC,
        risk_target_daily_vol=RISK_TARGET_DAILY_VOL,
        risk_max_name_exposure_frac=RISK_MAX_NAME_EXPOSURE_FRAC,
        risk_use_vol_scaling=RISK_USE_VOL_SCALING,
        risk_vol_lookback_days=RISK_VOL_LOOKBACK_DAYS,
        risk_vol_floor=RISK_VOL_FLOOR,
        risk_max_option_oi_frac=RISK_MAX_OPTION_OI_FRAC,
        risk_max_contracts_per_trade=RISK_MAX_CONTRACTS_PER_TRADE,
        drawdown_penalty=0.25,
        suppress_run_output=True,
    )
    best_cfg = _btu.select_best_backtest_config(
        strict_grid_df,
        strict_only=True,
        max_stop_loss_frac=0.10,
        min_entry_threshold=0.0010,
        min_hold_days=2,
        max_hold_days=10,
        min_exits=20,
        min_avg_days_held=2.0,
        prefer_fast_exit=True,
        prefer_tighter_stop=True,
        prefer_higher_threshold=True,
    )
    if best_cfg is None:
        raise ValueError("No strict config met constraints. Relax strict filters.")
    MAX_HOLD_BARS = int(best_cfg["hold_days"])
    STOP_LOSS_FRAC = float(best_cfg["stop_loss_frac"])
    ENTRY_PREDICTION_THRESHOLD = float(best_cfg["entry_threshold"])
    DTE_MIN = int(best_cfg["dte_min"])
    DTE_MAX = int(best_cfg["dte_max"])
    EXIT_DTE_THRESHOLD = int(best_cfg["exit_dte_threshold"])
    strict_grid_path = ROOT / "tempfolder" / "strict_dynamic_optimization_new_predictions.csv"
    strict_grid_df.to_csv(strict_grid_path, index=False)
    print("[opt] selected strict params:")
    print({
        "MAX_HOLD_BARS": MAX_HOLD_BARS,
        "STOP_LOSS_FRAC": STOP_LOSS_FRAC,
        "ENTRY_PREDICTION_THRESHOLD": ENTRY_PREDICTION_THRESHOLD,
        "DTE_MIN": DTE_MIN,
        "DTE_MAX": DTE_MAX,
        "EXIT_DTE_THRESHOLD": EXIT_DTE_THRESHOLD,
    })
    print(f"[opt] grid saved: {strict_grid_path}")

results = _btu.run_backtest(
    predictions_df=predictions_df,
    options_df=options_df,
    K=K_LONG_BACKTEST,
    initial_capital=INITIAL_CAPITAL,
    max_holding_days=MAX_HOLD_BARS,
    exit_dte_threshold=EXIT_DTE_THRESHOLD,
    num_contracts=NUM_CONTRACTS,
    commission_per_contract=COMMISSION_PER_CONTRACT,
    half_spread_pct_stock=0.0005,
    half_spread_pct_option=0.02,
    dte_min=DTE_MIN,
    dte_max=DTE_MAX,
    entry_moneyness_min=ENTRY_MONEYNESS_MIN,
    entry_moneyness_max=ENTRY_MONEYNESS_MAX,
    entry_min_open_interest=ENTRY_MIN_OPEN_INTEREST,
    entry_max_spread_frac=ENTRY_MAX_SPREAD_FRAC,
    use_signal_exit=False,
    top_k_df=top_k_for_backtest,
    entry_prediction_threshold=ENTRY_PREDICTION_THRESHOLD,
    stop_loss_frac_of_entry_cost=STOP_LOSS_FRAC,
    earnings_cycle_mode=True,
    allow_short_signals=ALLOW_SHORT_SIGNALS,
    signal_side_col=SIGNAL_SIDE_COL,
    sizing_mode=SIZING_MODE,
    risk_target_gross_exposure_frac=RISK_TARGET_GROSS_EXPOSURE_FRAC,
    risk_target_daily_vol=RISK_TARGET_DAILY_VOL,
    risk_max_name_exposure_frac=RISK_MAX_NAME_EXPOSURE_FRAC,
    risk_use_vol_scaling=RISK_USE_VOL_SCALING,
    risk_vol_lookback_days=RISK_VOL_LOOKBACK_DAYS,
    risk_vol_floor=RISK_VOL_FLOOR,
    risk_max_option_oi_frac=RISK_MAX_OPTION_OI_FRAC,
    risk_max_contracts_per_trade=RISK_MAX_CONTRACTS_PER_TRADE,
)


In [ ]:
# --- Daily P&L and position count over time ---
daily_df     = results["daily_pnl_df"]
trade_log    = results["trade_log"]
position_log = results["position_log"]
trade_book  = results.get("trade_book", pd.DataFrame())

fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

cum_pnl = daily_df["daily_pnl"].cumsum()
axes[0].plot(cum_pnl.index, cum_pnl.values, color="steelblue", linewidth=1.5)
axes[0].axhline(0, color="black", linewidth=0.5)
axes[0].set_ylabel("Cumulative P&L ($)")
axes[0].set_title("Cumulative Portfolio P&L", fontsize=12)
axes[0].fill_between(cum_pnl.index, cum_pnl.values, 0,
                     where=cum_pnl.values >= 0, alpha=0.15, color="green")
axes[0].fill_between(cum_pnl.index, cum_pnl.values, 0,
                     where=cum_pnl.values < 0, alpha=0.15, color="red")

axes[1].bar(daily_df.index, daily_df["daily_pnl"],
            color=["seagreen" if v >= 0 else "tomato" for v in daily_df["daily_pnl"]],
            width=1.5, alpha=0.8)
axes[1].axhline(0, color="black", linewidth=0.5)
axes[1].set_ylabel("Daily P&L ($)")
axes[1].set_title("Daily P&L", fontsize=12)

axes[2].plot(daily_df.index, daily_df["n_open_positions"],
             color="darkorange", linewidth=1, drawstyle="steps-post")
axes[2].set_ylabel("Open Positions")
axes[2].set_xlabel("Date")
axes[2].set_title("Number of Open Positions", fontsize=12)
axes[2].set_ylim(0, K_LONG_BACKTEST + (K_SHORT_BACKTEST if ALLOW_SHORT_SIGNALS else 0) + 1)

for ax in axes:
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    ax.xaxis.set_major_locator(mdates.YearLocator())

plt.tight_layout(); plt.show()

print("\nExit reason breakdown:")
if not trade_log.empty:
    exits = trade_log[trade_log["action"] == "exit"]
    print(exits["exit_reason"].value_counts().to_string())

print(f"Trade log rows: {len(trade_log)} | Round-trip trade book rows: {len(trade_book)}")



In [ ]:
# --- Export backtest results to CSV ---
os.makedirs("data/results", exist_ok=True)

cum_pnl = daily_df["daily_pnl"].cumsum().rename("cumulative_pnl")
export_daily = daily_df.join(cum_pnl)
export_daily.to_csv("data/results/daily_pnl.csv")

trade_log.to_csv("data/results/trade_log.csv", index=False)
position_log.to_csv("data/results/position_log.csv", index=False)
if "trade_book" in dir() and isinstance(trade_book, pd.DataFrame):
    trade_book.to_csv("data/results/trade_book.csv", index=False)

print("Saved:")
print(f"  data/results/daily_pnl.csv     — {len(export_daily)} rows")
print(f"  data/results/trade_log.csv     — {len(trade_log)} rows")
print(f"  data/results/position_log.csv  — {len(position_log)} rows")
if "trade_book" in dir() and isinstance(trade_book, pd.DataFrame):
    print(f"  data/results/trade_book.csv    — {len(trade_book)} rows")
print(f"  INITIAL_CAPITAL:                ${INITIAL_CAPITAL:,.0f}")
print(f"  NUM_CONTRACTS:                  {NUM_CONTRACTS}")



In [ ]:
# --- P&L attribution: option vs stock hedge ---
if not position_log.empty:
    pos_log = pd.DataFrame(position_log) if isinstance(position_log, list) else position_log
    daily_attr = pos_log.groupby("date")[["option_pnl", "stock_pnl", "daily_pnl"]].sum()

    fig, ax = plt.subplots(figsize=(14, 5))
    daily_attr["option_pnl"].cumsum().plot(ax=ax, label="Option P&L", color="steelblue")
    daily_attr["stock_pnl"].cumsum().plot(ax=ax, label="Stock Hedge P&L", color="tomato")
    daily_attr["daily_pnl"].cumsum().plot(ax=ax, label="Total P&L", color="black", linewidth=2)
    ax.axhline(0, color="gray", linewidth=0.5)
    ax.set_title("Stage 12 — P&L Attribution: Option vs Delta Hedge", fontsize=13)
    ax.set_ylabel("Cumulative P&L ($)")
    ax.legend(); plt.tight_layout(); plt.show()


---
## Stage 13 — Performance Evaluation (`src/performance.py`)

Key metrics computed from the strategy equity curve and compared to three benchmarks:

1. **SPY Buy & Hold** — passive market benchmark on the same date window
2. **Equal-Weight Universe** — equal-weight portfolio across all prediction tickers, rebalanced daily
3. **Delta-Equivalent SPY** *(v2 addition)* — SPY position sized to match the strategy's net delta
   exposure on each day. Only active on days the strategy has open positions. This controls for
   market beta and makes the alpha attributable to option selection and earnings timing,
   not just residual directional exposure.

**v2 note on annualisation validity:**  
The strategy's `hit_rate` and Sharpe ratio are computed over the full 2010–2024 sample.
With an earnings-triggered strategy (~4 trades/quarter × 8 tickers = ~96 trades/year),
the number of *independent observations* is much smaller than the 3,500+ calendar days.
The annualised Sharpe computed from daily returns overstates precision; interpret it alongside
the trade-level win rate and reward/risk ratio from Stage 10 as the primary edge metrics.


In [ ]:
eq = results["equity_curve"]
dd = results["drawdown"]

fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

axes[0].plot(eq.index, eq.values, color="steelblue", linewidth=1.5, label="Strategy (v2)")
axes[0].axhline(INITIAL_CAPITAL, color="gray", linewidth=0.8, linestyle="--", label="Initial Capital")
axes[0].set_ylabel("Portfolio Value ($)")
axes[0].set_title("Stage 13 — Strategy Equity Curve (v2)", fontsize=13)
axes[0].legend()

axes[1].fill_between(dd.index, dd.values * 100, 0, color="tomato", alpha=0.7)
axes[1].set_ylabel("Drawdown (%)")
axes[1].set_xlabel("Date")
axes[1].set_title("Drawdown from Peak", fontsize=12)

for ax in axes:
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    ax.xaxis.set_major_locator(mdates.YearLocator())

plt.tight_layout(); plt.show()


In [ ]:
# ── Standard benchmarks (SPY + equal-weight) ──────────────────────────────
universe_tickers = sorted(predictions_df["ticker"].unique().tolist())

eval_results = evaluate_performance(
    results=results,
    prices_df=prices_df,
    universe_tickers=universe_tickers,
)

# ── Improvement 5: delta-equivalent SPY benchmark ─────────────────────────
# Build a SPY position sized daily to match the strategy's net dollar delta exposure.
# This benchmark is only "invested" on days the strategy has open positions.
_pos_log = results["position_log"] if isinstance(results["position_log"], pd.DataFrame)            else pd.DataFrame(results["position_log"])

if not _pos_log.empty and "SPY" in prices_df["ticker"].unique():
    _spy_px = (
        prices_df[prices_df["ticker"] == "SPY"]
        .set_index("date")["adj_close"]
        .sort_index()
    )
    # Net dollar delta per day: sum(delta * 100 * stock_price) across open positions
    _pos_log["date"] = pd.to_datetime(_pos_log["date"])
    _pos_log["dollar_delta"] = _pos_log["delta"] * CONTRACT_SIZE * _pos_log["stock_price"]
    _daily_delta = _pos_log.groupby("date")["dollar_delta"].sum()

    # Build delta-equivalent SPY equity curve
    _spy_ret = _spy_px.pct_change().dropna()
    _delta_eq_pnl = pd.Series(0.0, index=eq.index)
    for _date, _net_delta in _daily_delta.items():
        if _date in _spy_ret.index:
            _delta_eq_pnl[_date] = _net_delta * float(_spy_ret.loc[_date])
    _delta_spy_eq = (INITIAL_CAPITAL + _delta_eq_pnl.cumsum()).rename("delta_spy_equity")
    eval_results["benchmark_equities"]["Delta-Equivalent SPY"] = _delta_spy_eq
    print("Delta-equivalent SPY benchmark constructed.")
else:
    print("Skipping delta-equivalent SPY benchmark (position_log empty or SPY not in prices_df).")

print("\n=== Performance Table (v2) ===")
display(eval_results["performance_table"])


In [ ]:
benchmark_equities = eval_results.get("benchmark_equities", {})

fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(eq.index, eq.values, label="Strategy v2 (Delta-Hedged)", color="steelblue", linewidth=2)

_colors = {"SPY Buy & Hold": "darkorange", "Equal-Weight Universe": "seagreen",
           "Delta-Equivalent SPY": "purple"}
_styles = {"SPY Buy & Hold": "--", "Equal-Weight Universe": "--", "Delta-Equivalent SPY": ":"}
for label, bench_eq in benchmark_equities.items():
    if not bench_eq.empty:
        aligned = bench_eq.reindex(eq.index, method="ffill")
        ax.plot(aligned.index, aligned.values, label=label,
                color=_colors.get(label, "gray"),
                linestyle=_styles.get(label, "--"),
                linewidth=1.5, alpha=0.8)

ax.axhline(INITIAL_CAPITAL, color="gray", linewidth=0.6, linestyle=":")
ax.set_title("Stage 13 — Strategy vs Benchmarks (v2)", fontsize=13)
ax.set_ylabel("Portfolio Value ($)"); ax.set_xlabel("Date")
ax.legend(); ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
ax.xaxis.set_major_locator(mdates.YearLocator())
plt.tight_layout(); plt.show()


In [ ]:
returns = eq.pct_change().dropna()

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].hist(returns * 100, bins=60, color="steelblue", edgecolor="white", alpha=0.85)
axes[0].axvline(0, color="black", linewidth=0.8)
axes[0].axvline(float(returns.mean() * 100), color="red", linewidth=1.5, linestyle="--",
                label=f"Mean: {returns.mean()*100:.3f}%")
axes[0].set_title("Daily Return Distribution")
axes[0].set_xlabel("Daily Return (%)"); axes[0].legend()

# Rolling Sharpe — 63-day window (one quarter)
rf_daily = (1 + 0.04) ** (1 / 252) - 1
rolling_sharpe = (
    (returns - rf_daily).rolling(63).mean()
    / returns.rolling(63).std()
    * (252 ** 0.5)
).replace([np.inf, -np.inf], np.nan)
axes[1].plot(rolling_sharpe.index, rolling_sharpe.values, color="darkorange", linewidth=1)
axes[1].axhline(0, color="black", linewidth=0.6)
axes[1].set_title("Rolling 63-Day Sharpe Ratio")
axes[1].set_xlabel("Date")
axes[1].xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
axes[1].xaxis.set_major_locator(mdates.YearLocator())

plt.suptitle("Stage 13 — Return Analysis (v2)", fontsize=13)
plt.tight_layout(); plt.show()

m = results["metrics"]
print("\n=== Strategy Metrics (v2) ===")
for k, v in m.items():
    if k == "n_days":
        print(f"  {k:<20}: {int(v)}")
    elif isinstance(v, float) and not pd.isna(v):
        if k in ("total_return", "cagr", "ann_volatility", "max_drawdown", "hit_rate"):
            print(f"  {k:<20}: {v:.2%}")
        else:
            print(f"  {k:<20}: {v:.4f}")
    else:
        print(f"  {k:<20}: {v}")

print("\nNote: Sharpe annualisation assumes i.i.d. daily returns.")
print("With ~96 trades/year, the effective sample for inference is trade-level,")
print("not the daily return count. See Stage 10 for trade-level edge metrics.")


---
## Trade Log & Position Diagnostics


In [ ]:
if not trade_log.empty:
    entries = trade_log[trade_log["action"] == "enter"].copy()
    exits   = trade_log[trade_log["action"] == "exit"].copy()

    print(f"Total entries: {len(entries)}  |  Total exits: {len(exits)}")

    if not exits.empty:
        print(f"\nRealized P&L on exits (first 20):")
        print(exits[["date", "ticker", "exit_reason", "days_held", "realized_pnl", "exit_cost"]]
              .sort_values("date").head(20).to_string(index=False))

        print(f"\nP&L by exit reason:")
        print(exits.groupby("exit_reason")["realized_pnl"].agg(["count", "sum", "mean"]).round(2))

        print(f"\nP&L by ticker:")
        print(exits.groupby("ticker")["realized_pnl"].agg(["count", "sum", "mean"]).round(2))

        fig, ax = plt.subplots(figsize=(8, 4))
        ax.hist(exits["days_held"].dropna(), bins=20, color="steelblue", edgecolor="white")
        ax.set_title("Holding Period Distribution (Days)")
        ax.set_xlabel("Days Held"); ax.set_ylabel("Count")
        plt.tight_layout(); plt.show()


In [ ]:
# ============================================================
# Full Trade Book — Every Entry, Exit, and Position Construction
# ============================================================
entries = trade_log[trade_log["action"] == "enter"].copy().reset_index(drop=True)
exits   = trade_log[trade_log["action"] == "exit"].copy().reset_index(drop=True)

exit_lookup = {}
for _, ex in exits.iterrows():
    key = (ex["ticker"], ex.get("entry_date", ex["date"]))
    exit_lookup[key] = ex

rows = []
for _, en in entries.iterrows():
    ticker     = en["ticker"]
    entry_date = en["date"]
    strike     = en.get("strike", float("nan"))
    expiry     = en.get("expiry", pd.NaT)
    dte        = en.get("dte", float("nan"))
    entry_opt  = en.get("option_price", float("nan"))
    entry_stk  = en.get("stock_price", float("nan"))
    delta      = en.get("delta", float("nan"))
    stock_pos  = en.get("stock_position", float("nan"))
    rank       = en.get("rank", float("nan"))
    pred       = en.get("prediction", float("nan"))
    entry_cost = en.get("entry_cost", float("nan"))

    hedge_shares = abs(stock_pos) if not pd.isna(stock_pos) else float("nan")
    construction = (
        f"Long {NUM_CONTRACTS}c  K={strike:.0f}"
        f"  exp={pd.Timestamp(expiry).date() if pd.notna(expiry) else 'n/a'}"
        f"  DTE={int(dte) if not pd.isna(dte) else 'n/a'}"
        f"  | Short {hedge_shares:.0f}sh @ ${entry_stk:.2f}  (δ={delta:.3f})"
        if not pd.isna(strike) else "n/a"
    )

    ex_row = exit_lookup.get((ticker, entry_date), None)
    if ex_row is not None:
        exit_date    = ex_row["date"]
        exit_reason  = ex_row.get("exit_reason", "")
        days_held    = ex_row.get("days_held", float("nan"))
        realized_pnl = ex_row.get("realized_pnl", float("nan"))
        exit_cost_v  = ex_row.get("exit_cost", float("nan"))
    else:
        exit_date = exit_reason = pd.NaT
        days_held = realized_pnl = exit_cost_v = float("nan")

    rows.append({
        "signal_date":    en.get("signal_date", pd.NaT),
        "entry_date":     entry_date,
        "exit_date":      exit_date,
        "ticker":         ticker,
        "rank":           rank,
        "prediction":     round(pred, 5) if not pd.isna(pred) else float("nan"),
        "strike":         strike,
        "expiry":         pd.Timestamp(expiry).date() if pd.notna(expiry) else pd.NaT,
        "dte_entry":      int(dte) if not pd.isna(dte) else float("nan"),
        "delta_entry":    round(delta, 3) if not pd.isna(delta) else float("nan"),
        "n_contracts":    NUM_CONTRACTS,
        "entry_opt_$":    round(entry_opt, 2) if not pd.isna(entry_opt) else float("nan"),
        "entry_stk_$":    round(entry_stk, 2) if not pd.isna(entry_stk) else float("nan"),
        "stock_pos":      int(stock_pos * NUM_CONTRACTS) if not pd.isna(stock_pos) else float("nan"),
        "entry_cost_$":   round(entry_cost, 2) if not pd.isna(entry_cost) else float("nan"),
        "days_held":      int(days_held) if not pd.isna(days_held) else float("nan"),
        "exit_reason":    exit_reason,
        "exit_cost_$":    round(exit_cost_v, 2) if not pd.isna(exit_cost_v) else float("nan"),
        "realized_pnl_$": round(realized_pnl, 2) if not pd.isna(realized_pnl) else float("nan"),
        "construction":   construction,
    })

trade_book = pd.DataFrame(rows).sort_values("entry_date").reset_index(drop=True)

print(f"Total trades: {len(trade_book)}  |  "
      f"Closed: {trade_book['exit_date'].notna().sum()}  |  "
      f"Open/unmatched: {trade_book['exit_date'].isna().sum()}")
print(f"Total realized P&L: ${trade_book['realized_pnl_$'].sum():,.2f}")
print(f"Win rate: {(trade_book['realized_pnl_$'] > 0).mean():.1%}")
print()

display_cols = [
    "entry_date", "exit_date", "ticker", "rank", "prediction",
    "strike", "expiry", "dte_entry", "delta_entry", "n_contracts",
    "entry_opt_$", "entry_stk_$", "stock_pos", "entry_cost_$",
    "days_held", "exit_reason", "exit_cost_$", "realized_pnl_$",
]
display(trade_book[display_cols].style
    .format(na_rep="—")
    .applymap(lambda v: "color: green" if isinstance(v, (int, float)) and v > 0 else
                        ("color: red" if isinstance(v, (int, float)) and v < 0 else ""),
              subset=["realized_pnl_$"])
    .set_caption(f"Full Trade Book — {NUM_CONTRACTS} contract(s) per position"))

print("\n=== Position Construction Detail (first 20) ===")
for _, r in trade_book.head(20).iterrows():
    pnl_str = f"  →  PnL: ${r['realized_pnl_$']:+.2f}" if not pd.isna(r["realized_pnl_$"]) else "  →  (open)"
    print(f"  [{r['entry_date'].date() if pd.notna(r['entry_date']) else 'n/a'}] "
          f"{r['ticker']:<6}  {r['construction']}{pnl_str}")


---
## Stage 10 — Risk Management Framework (`src/risk_utils.py` + `src/exposure_utils.py`)

**Purpose:** Portfolio risk controls and exposure monitoring from realized backtest positions.
All metrics are derived from `trade_log`, `position_log`, and `daily_pnl_df` — never from
raw signals. This ensures risk figures reflect actual executed trades.

**Metrics computed:**
- Max drawdown, drawdown series, start / recovery date and duration, N distinct periods
- Rolling 21-day Sharpe, Sortino, and realized volatility (inf-safe)
- Historical VaR and CVaR (Expected Shortfall) at 0.1%, 1%, 5%, 10% confidence levels
- Gross vs net dollar exposure and net delta exposure (from position_log or trade_log reconstruction)
- Portfolio beta to SPY (delta-weighted; pass `beta_lookup` for per-ticker betas)
- Concentration: HHI and max single-ticker weight
- Trade-level: win rate, reward/risk ratio, exit reason breakdown, holding period distribution
- Drawdown-triggered (−10%) and volatility-spike (2× rolling median) de-risking event log

**Outputs:** `data/results/risk_exposure_daily.csv`, `data/results/risk_events.csv`

**Risk considerations addressed (from project risk analysis):**
drawdown analysis, CVaR/ES, Sortino decomposition, gross vs net exposure,
portfolio beta, concentration (HHI), stop-loss calibration, holding period diagnostics,
drawdown-triggered de-risking events, and annualisation validity caveats.


In [ ]:
# ============================================================
# Stage 10 — Risk Management Framework (v2)
# Uses live results dict from run_backtest.
# Falls back to saved CSVs if kernel was restarted.
# Outputs: data/results/risk_exposure_daily.csv
#          data/results/risk_events.csv
# ============================================================

importlib.reload(src.risk_utils); importlib.reload(src.exposure_utils)
from src.risk_utils import compute_all_risk_metrics
from src.exposure_utils import run_stage10, build_risk_exposure_daily

_nb_dir       = Path(".").resolve()
RESULTS_DIR   = str(_nb_dir / "data" / "results")
TRADE_LOG_PATH    = str(_nb_dir / "data" / "results" / "trade_log.csv")
POSITION_LOG_PATH = str(_nb_dir / "data" / "results" / "position_log.csv")
DAILY_PNL_PATH    = str(_nb_dir / "data" / "results" / "daily_pnl.csv")

if "results" in dir() and results is not None:
    print("[Stage 10] Using live results dict from run_backtest.")
    _equity       = results["equity_curve"]
    _position_log = (results["position_log"]
                     if isinstance(results["position_log"], pd.DataFrame)
                     else pd.DataFrame(results["position_log"]))
    _trade_log    = (results["trade_log"]
                     if isinstance(results["trade_log"], pd.DataFrame)
                     else pd.DataFrame(results["trade_log"]))
    _daily_pnl    = results["daily_pnl_df"].reset_index() if "daily_pnl_df" in results else None

    _raw = compute_all_risk_metrics(
        trade_log=_trade_log,
        position_log=_position_log if not _position_log.empty else None,
        daily_pnl_df=_daily_pnl,
        beta_lookup=None,            # swap in {ticker: beta} when available
        sector_lookup=None,          # swap in {ticker: sector} when available
        initial_capital=INITIAL_CAPITAL,
        drawdown_threshold=-0.10,
        max_gross_exposure_pct=0.10,
        max_net_exposure_pct=0.05,
        max_abs_beta_exposure=0.25,
        max_single_name_weight=0.50,
        max_sector_weight=0.60,
    )
    _risk_exp_df = build_risk_exposure_daily(_raw)

    os.makedirs(RESULTS_DIR, exist_ok=True)
    _risk_exp_path = os.path.join(RESULTS_DIR, "risk_exposure_daily.csv")
    _risk_evt_path = os.path.join(RESULTS_DIR, "risk_events.csv")
    _stress_path = os.path.join(RESULTS_DIR, "stress_scenarios.csv")
    _risk_exp_df.to_csv(_risk_exp_path, index=False)
    _raw["risk_events"].to_csv(_risk_evt_path, index=False)
    _raw["stress_table"].to_csv(_stress_path, index=False)
    print(f"  risk_exposure_daily.csv  ({len(_risk_exp_df)} rows)")
    print(f"  risk_events.csv          ({len(_raw['risk_events'])} rows)")
    print(f"  stress_scenarios.csv     ({len(_raw['stress_table'])} rows)")

    stage10 = {
        "risk_exposure_daily": _risk_exp_df,
        "risk_events":         _raw["risk_events"],
        "trade_risk_stats":    _raw["trade_risk_stats"],
        "var_cvar_table":      _raw["var_cvar_table"],
        "drawdown_stats":      _raw["drawdown_stats"],
        "stress_table":        _raw["stress_table"],
        "paths": {
            "risk_exposure_path": _risk_exp_path,
            "risk_events_path":   _risk_evt_path,
            "stress_path":        _stress_path,
        }
    }
else:
    print("[Stage 10] 'results' not in scope — loading from saved CSVs.")
    stage10 = run_stage10(
        trade_log_path=TRADE_LOG_PATH,
        position_log_path=POSITION_LOG_PATH if os.path.exists(POSITION_LOG_PATH) else None,
        daily_pnl_path=DAILY_PNL_PATH       if os.path.exists(DAILY_PNL_PATH)    else None,
        output_dir=RESULTS_DIR,
        beta_lookup=None,
        sector_lookup=None,
        initial_capital=INITIAL_CAPITAL if "INITIAL_CAPITAL" in dir() else 1_000_000.0,
        drawdown_threshold=-0.10,
        max_gross_exposure_pct=0.10,
        max_net_exposure_pct=0.05,
        max_abs_beta_exposure=0.25,
        max_single_name_weight=0.50,
        max_sector_weight=0.60,
    )

risk_exp    = stage10["risk_exposure_daily"]
risk_evts   = stage10["risk_events"]
var_tbl     = stage10["var_cvar_table"]
dd_stats    = stage10["drawdown_stats"]
trade_stats = stage10["trade_risk_stats"]
stress_tbl  = stage10.get("stress_table", pd.DataFrame())
dd_series   = dd_stats["drawdown_series"]

# ── Printed risk summary ─────────────────────────────────────────────────────
print("\n" + "=" * 64)
print("STAGE 10 — RISK SUMMARY (v2)")
print("=" * 64)

print("\n── Drawdown Statistics ──")
print(f"  Max drawdown:           {dd_stats['max_drawdown']:.2%}")
print(f"  Max drawdown date:      {dd_stats['max_drawdown_date']}")
print(f"  Drawdown started:       {dd_stats['drawdown_start_date']}")
print(f"  Recovery date:          {dd_stats['recovery_date'] or 'Not recovered in sample'}")
print(f"  Recovery days:          {dd_stats['recovery_days'] or 'N/A'}")
print(f"  N distinct DD periods:  {dd_stats['n_drawdown_periods']}")

print("\n── VaR and CVaR (Historical Simulation — daily returns) ──")
vc = var_tbl.copy()
vc["confidence_level"] = vc["confidence_level"].map(lambda x: f"{x:.1%}")
vc["var_pct"]    = vc["var_pct"].map(lambda x: f"{x:.4%}")
vc["cvar_pct"]   = vc["cvar_pct"].map(lambda x: f"{x:.4%}")
vc["var_dollar"]  = vc["var_dollar"].map(lambda x: f"${x:,.0f}")
vc["cvar_dollar"] = vc["cvar_dollar"].map(lambda x: f"${x:,.0f}")
print(vc.to_string(index=False))

print("\n── Rolling Risk (21-day window) ──")
if "rolling_sharpe_21d" in risk_exp.columns:
    rs = risk_exp["rolling_sharpe_21d"].replace([np.inf, -np.inf], np.nan).dropna()
    rv = risk_exp["rolling_vol_21d"].replace([np.inf, -np.inf], np.nan).dropna()
    so = risk_exp["rolling_sortino_21d"].replace([np.inf, -np.inf], np.nan).dropna()
    print(f"  Median Sharpe  (21d):   {rs.median():.3f}")
    print(f"  Median Sortino (21d):   {so.median():.3f}")
    print(f"  Median Ann. Vol:        {rv.median():.3f}")

print("\n── Exposure & Concentration ──")
exp_active = risk_exp[risk_exp["n_positions"].fillna(0) > 0]
if not exp_active.empty and "gross_exposure" in exp_active.columns:
    print(f"  Days with open positions: {len(exp_active)}")
    print(f"  Peak gross exposure:      ${exp_active['gross_exposure'].max():,.0f}")
    print(f"  Avg net exposure:         ${exp_active['net_exposure'].mean():,.0f}")
if "hhi" in exp_active.columns:
    print(f"  Avg HHI:                  {exp_active['hhi'].mean():.3f}  (1.0 = single position)")
    print(f"  Max single-ticker weight: {exp_active['max_weight_pct'].max():.1%}")
if "net_beta_exposure" in exp_active.columns:
    print(f"  Avg net beta to SPY:      {exp_active['net_beta_exposure'].mean():.3f}")

print("\n── Trade Risk Diagnostics ──")
ts = trade_stats
print(f"  N entries / exits:      {ts['n_entries']} / {ts['n_exits']}")
print(f"  Win rate:               {ts['win_rate']:.2%}")
print(f"  Avg win PnL:            ${ts['avg_win_pnl']:,.2f}")
print(f"  Avg loss PnL:           ${ts['avg_loss_pnl']:,.2f}")
print(f"  Total realized PnL:     ${ts['total_realized_pnl']:,.2f}")
rr = ts["reward_risk_ratio"]
print(f"  Reward/risk ratio:      {rr:.3f}" if not (isinstance(rr, float) and rr != rr) else "  Reward/risk ratio:      N/A")
print(f"  Stop-loss rate:         {ts['stop_loss_rate']:.2%}  ({ts['n_stop_loss']} trades)")
print(f"  Exit reasons:           {ts['exit_reason_counts']}")
print(f"  Avg days held:          {ts['avg_days_held']:.1f}")
print(f"  P25/P50/P75 days:       {ts['p25_days_held']:.0f} / {ts['p50_days_held']:.0f} / {ts['p75_days_held']:.0f}")
print(f"  Avg entry cost:         ${ts['avg_entry_cost']:,.2f}")

print("\n── Marked Exposure Limits ──")
for _col, _label in [
    ("gross_exposure_pct_equity", "Peak gross / equity"),
    ("net_exposure_pct_equity",   "Peak |net| / equity"),
    ("net_beta_exposure_pct_equity", "Peak |beta| / equity"),
]:
    if _col in risk_exp.columns:
        print(f"  {_label:<24}: {risk_exp[_col].max():.2%}")
for _flag, _label in [
    ("gross_exposure_limit_breach", "Gross exposure breaches"),
    ("net_exposure_limit_breach",   "Net exposure breaches"),
    ("beta_limit_breach",           "Beta exposure breaches"),
    ("single_name_limit_breach",    "Single-name breaches"),
    ("sector_limit_breach",         "Sector breaches"),
]:
    if _flag in risk_exp.columns:
        print(f"  {_label:<24}: {int(risk_exp[_flag].fillna(False).sum())}")

if not stress_tbl.empty:
    print("\n── Stress Scenarios (latest marked book) ──")
    print(stress_tbl.to_string(index=False))

if not risk_evts.empty:
    n_dd  = (risk_evts["trigger_type"] == "drawdown_breach").sum()
    n_vol = (risk_evts["trigger_type"] == "volatility_spike").sum()
    print(f"\n── Risk Events ({len(risk_evts)} total: {n_dd} drawdown breaches, {n_vol} vol spikes) ──")
    print(risk_evts[["date", "trigger_type", "value", "description"]].head(10).to_string(index=False))

# ── 6-panel risk dashboard ────────────────────────────────────────────────────
fig, axes = plt.subplots(3, 2, figsize=(16, 13))
fig.suptitle("Stage 10 — Risk Management Dashboard (v2)", fontsize=14, fontweight="bold", y=1.01)

risk_exp["date"] = pd.to_datetime(risk_exp["date"])
_dates = risk_exp["date"]
_eq    = risk_exp["equity"]
_dd    = (dd_series.reindex(risk_exp["date"]).fillna(0)
          if hasattr(dd_series, "reindex")
          else risk_exp.get("drawdown", pd.Series(0.0, index=risk_exp.index)).fillna(0))

# Panel 1 — Equity + drawdown breach markers
ax = axes[0, 0]
ax.plot(_dates, _eq, color="steelblue", linewidth=1.3, label="Strategy v2")
if not risk_evts.empty:
    for _, row in risk_evts[risk_evts["trigger_type"] == "drawdown_breach"].iterrows():
        ax.axvline(pd.to_datetime(row["date"]), color="crimson", alpha=0.35, linewidth=0.8)
ax.set_title("Equity Curve  (red lines = drawdown breach dates)")
ax.set_ylabel("Portfolio Value ($)")
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
ax.legend(fontsize=8); ax.grid(alpha=0.3)

# Panel 2 — Drawdown
ax = axes[0, 1]
ax.fill_between(_dates, _dd * 100, 0, color="crimson", alpha=0.55, label="Drawdown")
ax.axhline(-10, color="black", linestyle="--", linewidth=0.8, label="−10% threshold")
ax.set_title("Rolling Drawdown from Peak")
ax.set_ylabel("Drawdown (%)")
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
ax.legend(fontsize=8); ax.grid(alpha=0.3)

# Panel 3 — Rolling Sharpe & Sortino
ax = axes[1, 0]
if "rolling_sharpe_21d" in risk_exp.columns:
    _rs = risk_exp.set_index("date")["rolling_sharpe_21d"].replace([np.inf, -np.inf], np.nan)
    _so = risk_exp.set_index("date")["rolling_sortino_21d"].replace([np.inf, -np.inf], np.nan)
    ax.plot(_rs.index, _rs, color="steelblue", linewidth=0.85, label="Sharpe 21d")
    ax.plot(_so.index, _so, color="darkorange", linewidth=0.85, alpha=0.8, label="Sortino 21d")
    ax.axhline(0, color="black", linewidth=0.6)
ax.set_title("Rolling 21-Day Risk-Adjusted Returns")
ax.set_ylabel("Ratio (annualized)")
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
ax.legend(fontsize=8); ax.grid(alpha=0.3)

# Panel 4 — Gross vs Net Exposure
ax = axes[1, 1]
_exp_a = risk_exp[risk_exp["n_positions"].fillna(0) > 0].copy()
if not _exp_a.empty and "gross_exposure" in _exp_a.columns:
    ax.bar(_exp_a["date"], _exp_a["gross_exposure"], color="steelblue", alpha=0.6, width=4, label="Gross")
    ax.bar(_exp_a["date"], _exp_a["net_exposure"],   color="crimson",   alpha=0.55, width=4, label="Net")
ax.set_title("Gross vs Net Dollar Exposure (active days)")
ax.set_ylabel("Exposure ($)")
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
ax.legend(fontsize=8); ax.grid(alpha=0.3)

# Panel 5 — VaR / CVaR
ax = axes[2, 0]
_raw_vc = stage10["var_cvar_table"]
_labels = _raw_vc["confidence_level"].map(lambda x: f"{x:.1%}").tolist()
_x = np.arange(len(_labels)); _w = 0.35
ax.bar(_x - _w/2, _raw_vc["var_pct"]  * 100, _w, color="steelblue", alpha=0.8, label="VaR")
ax.bar(_x + _w/2, _raw_vc["cvar_pct"] * 100, _w, color="crimson",   alpha=0.8, label="CVaR (ES)")
ax.set_xticks(_x); ax.set_xticklabels(_labels)
ax.set_title("Daily VaR and CVaR (Historical Simulation)")
ax.set_ylabel("Daily return (%)"); ax.set_xlabel("Tail probability α")
ax.legend(fontsize=8); ax.grid(alpha=0.3, axis="y")

# Panel 6 — Realized PnL distribution
ax = axes[2, 1]
_exits_pl = (_trade_log[_trade_log["action"] == "exit"]["realized_pnl"].dropna()
             if ("_trade_log" in dir() and not _trade_log.empty)
             else pd.Series(dtype=float))
if _exits_pl.empty and os.path.exists(TRADE_LOG_PATH):
    _exits_pl = pd.read_csv(TRADE_LOG_PATH)
    _exits_pl = _exits_pl[_exits_pl["action"] == "exit"]["realized_pnl"].dropna()
if not _exits_pl.empty:
    ax.hist(_exits_pl, bins=35, color="steelblue", alpha=0.75, edgecolor="white")
    ax.axvline(0,                   color="crimson",    linewidth=1.4, linestyle="--", label="Break-even")
    ax.axvline(_exits_pl.mean(),    color="darkorange", linewidth=1.4, linestyle="--",
               label=f"Mean ${_exits_pl.mean():,.0f}")
    ax.axvline(_exits_pl.median(),  color="seagreen",   linewidth=1.2, linestyle=":",
               label=f"Median ${_exits_pl.median():,.0f}")
ax.set_title("Realized PnL Distribution (exits)")
ax.set_xlabel("Realized PnL ($)"); ax.set_ylabel("Count")
ax.legend(fontsize=8); ax.grid(alpha=0.3, axis="y")

plt.tight_layout()
_fig_dir = _nb_dir / "docs" / "figures"
_fig_dir.mkdir(parents=True, exist_ok=True)
try:
    plt.savefig(str(_fig_dir / "stage10_risk_dashboard_v2.png"), dpi=120, bbox_inches="tight")
    print(f"\n[Stage 10] Dashboard saved → docs/figures/stage10_risk_dashboard_v2.png")
except Exception as _e:
    print(f"[Stage 10] Figure save skipped: {_e}")
plt.show()

print("\n[Stage 10] Output files:")
print(f"  {stage10['paths']['risk_exposure_path']}")
print(f"  {stage10['paths']['risk_events_path']}")
if "stress_path" in stage10["paths"]:
    print(f"  {stage10['paths']['stress_path']}")


In [ ]:
# ============================================================
# Trading Costs — Setup (OOA framework per Mykland §2–3)
# ============================================================
import importlib
import sys
if ".." not in sys.path:
    sys.path.insert(0, "..")

import src.cost_model as cost_model
importlib.reload(cost_model)
from src.cost_model import compute_trade_costs, compute_markout_pnl, compute_slippage, mark_trades

# Cost parameters — calibrated to retail/institutional options execution
COST_PARAMS = dict(
    half_spread_pct_option = 0.02,    # 2% half-spread on option mid (illiquid)
    half_spread_pct_stock  = 0.0005,  # 5 bps half-spread on stock (liquid large-cap)
    stock_order_side       = "aggressive",
    commission_per_contract = 1.00,   # $1/contract — typical retail (IBKR tiered)
    commission_per_share   = 0.005,   # $0.005/share — IBKR tiered
    min_stock_commission   = 1.00,    # $1 minimum per stock order
    num_contracts          = NUM_CONTRACTS,  # from backtest run cell
    contract_multiplier    = 100,
)

print("Cost model loaded.")
print(f"Option half-spread: {COST_PARAMS['half_spread_pct_option']:.1%}")
print(f"Stock  half-spread: {COST_PARAMS['half_spread_pct_stock']:.4%}")
print(f"Commission/contract: ${COST_PARAMS['commission_per_contract']:.2f}")


In [ ]:
# ============================================================
# Trading Costs — Single Trade Verification
# ============================================================
# Pick one representative trade from the trade_log to verify cost math.
_sample_entry = trade_log[trade_log["action"] == "enter"].iloc[0]

_opt_entry_cost = compute_trade_costs(
    trade_type="option_entry",
    quantity=COST_PARAMS["num_contracts"],
    price=float(_sample_entry["option_price"]),
    bid_ask_spread_frac=COST_PARAMS["half_spread_pct_option"] * 2,
    order_side="aggressive",
    num_contracts=COST_PARAMS["num_contracts"],
    commission_per_contract=COST_PARAMS["commission_per_contract"],
    contract_multiplier=COST_PARAMS["contract_multiplier"],
)

_stk_entry_cost = compute_trade_costs(
    trade_type="stock_entry",
    quantity=abs(float(_sample_entry["stock_position"])),
    price=float(_sample_entry["stock_price"]),
    bid_ask_spread_frac=COST_PARAMS["half_spread_pct_stock"] * 2,
    order_side=COST_PARAMS["stock_order_side"],
    commission_per_share=COST_PARAMS["commission_per_share"],
    min_stock_commission=COST_PARAMS["min_stock_commission"],
    contract_multiplier=1,
)

print("=== Single Trade Cost Breakdown ===")
print(f"Ticker:              {_sample_entry['ticker']}")
print(f"Option mid:          ${float(_sample_entry['option_price']):.2f}")
print(f"  Option spread cost: ${_opt_entry_cost['spread_cost']:.4f}")
print(f"  Option commission:  ${_opt_entry_cost['commission']:.4f}")
print(f"  Option total cost:  ${_opt_entry_cost['total_cost']:.4f}")
print(f"  Option cost (bps):  {_opt_entry_cost['cost_bps']:.2f} bps")
print()
print(f"Stock position:      {float(_sample_entry['stock_position']):.0f} shares")
print(f"Stock price:         ${float(_sample_entry['stock_price']):.2f}")
print(f"  Stock spread cost:  ${_stk_entry_cost['spread_cost']:.4f}")
print(f"  Stock commission:   ${_stk_entry_cost['commission']:.4f}")
print(f"  Stock total cost:   ${_stk_entry_cost['total_cost']:.4f}")
print()
print(f"Entry total cost:    ${_opt_entry_cost['total_cost'] + _stk_entry_cost['total_cost']:.4f}")

# EXPECTED OUTCOME:
# Option commission = $1.00 * NUM_CONTRACTS
# Stock cost = max($1.00, 0.005 * abs(shares))
# cost_bps for option: typically 200-400 bps at 2% half-spread
assert _opt_entry_cost["total_cost"] > 0, "Option entry cost must be positive"
assert _stk_entry_cost["total_cost"] > 0, "Stock entry cost must be positive"
assert _opt_entry_cost["cost_bps"] > 0, "Option bps must be positive"
print("\nAll cost assertions PASSED.")


In [ ]:
# ============================================================
# Trading Costs — Full Trade Log Decoration (OOA marks §2.2)
# ============================================================
marked_trades = mark_trades(
    trade_log=trade_log,
    **COST_PARAMS,
)

_exits_marked = marked_trades[marked_trades["action"] == "exit"].copy()

print(f"Marked trade log: {len(marked_trades)} rows")
print(f"Exit rows: {len(_exits_marked)}")
print()
print("=== Cost Summary (exit rows) ===")
print(f"  Total gross realized PnL:    ${_exits_marked['realized_pnl'].sum():>12,.2f}")
print(f"  Total trading costs:          ${_exits_marked['cost_total'].sum():>12,.2f}")
print(f"  Total net PnL after costs:    ${_exits_marked['net_pnl'].sum():>12,.2f}")
print(f"  Cost drag (bps of notional):  {_exits_marked['cost_bps_option'].mean():.1f} bps avg option")
print()
print(f"  Gross win rate: {(_exits_marked['realized_pnl'] > 0).mean():.1%}")
print(f"  Net win rate:   {(_exits_marked['net_pnl'] > 0).mean():.1%}")

_net_wins   = _exits_marked[_exits_marked["net_pnl"] > 0]["net_pnl"].sum()
_net_losses = _exits_marked[_exits_marked["net_pnl"] < 0]["net_pnl"].sum()
_pf_net = _net_wins / abs(_net_losses) if _net_losses != 0 else float("inf")
print(f"  Net Profit Factor: {_pf_net:.3f}")

assert (_exits_marked["cost_total"].dropna() >= 0).all(), "FAIL: negative cost found"
assert _exits_marked["net_pnl"].sum() <= _exits_marked["realized_pnl"].sum() + 1e-6, \
    "FAIL: net PnL should not exceed gross PnL"
print("\nAll trade decoration assertions PASSED.")


In [ ]:
# ============================================================
# Trading Costs — P&L Term Structure (OOA §4.1)
# ============================================================
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import numpy as np

_exits = marked_trades[marked_trades["action"] == "exit"].copy()
_exits["date"] = pd.to_datetime(_exits["date"])

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle("OOA — Trading Cost Analysis", fontsize=14, fontweight="bold")

# Panel 1: Gross vs Net PnL distribution
ax = axes[0, 0]
_bins = 35
ax.hist(_exits["realized_pnl"], bins=_bins, alpha=0.65, color="steelblue",
        label="Gross PnL", edgecolor="white")
ax.hist(_exits["net_pnl"].dropna(), bins=_bins, alpha=0.65, color="tomato",
        label="Net PnL (after costs)", edgecolor="white")
ax.axvline(0, color="black", linewidth=1)
ax.axvline(_exits["realized_pnl"].mean(), color="steelblue", linewidth=1.5,
           linestyle="--", label=f"Gross mean ${_exits['realized_pnl'].mean():,.0f}")
ax.axvline(_exits["net_pnl"].mean(), color="tomato", linewidth=1.5,
           linestyle="--", label=f"Net mean ${_exits['net_pnl'].mean():,.0f}")
ax.set_title("PnL Distribution: Gross vs Net of Costs")
ax.set_xlabel("PnL ($)"); ax.set_ylabel("Count")
ax.legend(fontsize=8)

# Panel 2: Cost breakdown by ticker
ax = axes[0, 1]
_cost_by_ticker = _exits.groupby("ticker")["cost_total"].mean().sort_values(ascending=False)
ax.bar(_cost_by_ticker.index, _cost_by_ticker.values, color="darkorange", edgecolor="white")
ax.set_title("Average Cost per Trade by Ticker")
ax.set_xlabel("Ticker"); ax.set_ylabel("Avg Total Cost ($)")

# Panel 3: Cumulative gross vs net PnL over time
ax = axes[1, 0]
_exits_sorted = _exits.sort_values("date")
_cum_gross = _exits_sorted["realized_pnl"].cumsum()
_cum_net   = _exits_sorted["net_pnl"].cumsum()
ax.plot(_exits_sorted["date"], _cum_gross, color="steelblue", linewidth=1.5,
        label="Gross cumulative PnL")
ax.plot(_exits_sorted["date"], _cum_net,   color="tomato",    linewidth=1.5,
        label="Net cumulative PnL")
ax.fill_between(_exits_sorted["date"], _cum_gross, _cum_net,
                alpha=0.25, color="orange", label="Cost drag")
ax.axhline(0, color="black", linewidth=0.5)
ax.set_title("Cumulative PnL: Gross vs Net")
ax.set_xlabel("Date"); ax.set_ylabel("Cumulative PnL ($)")
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
ax.xaxis.set_major_locator(mdates.YearLocator())
ax.legend(fontsize=8)

# Panel 4: Category split — order type cost impact
ax = axes[1, 1]
if "m_type" in _exits.columns and _exits["m_type"].notna().any():
    _cat = _exits.groupby("m_type", dropna=True)[["realized_pnl", "net_pnl"]].mean()
    x = np.arange(len(_cat)); w = 0.35
    ax.bar(x - w/2, _cat["realized_pnl"], w, color="steelblue", alpha=0.8, label="Gross PnL")
    ax.bar(x + w/2, _cat["net_pnl"],      w, color="tomato",    alpha=0.8, label="Net PnL")
    ax.set_xticks(x); ax.set_xticklabels(_cat.index)
    ax.set_title("Avg PnL by Order Type (Passive/Aggressive)")
    ax.set_ylabel("Avg PnL ($)")
    ax.legend(fontsize=8)
else:
    ax.text(0.5, 0.5, "m_type column not available", ha="center", va="center",
            transform=ax.transAxes)

plt.tight_layout()
_fig_dir = Path("..") / "docs" / "figures"
_fig_dir.mkdir(parents=True, exist_ok=True)
plt.savefig(str(_fig_dir / "ooa_cost_analysis.png"), dpi=120, bbox_inches="tight")
plt.show()
print("OOA cost analysis figure saved.")


---
## Point-in-Time (PIT) Analysis

Ensures no-lookahead bias: at every decision date `t`, only information
available at close of `t` is used. Three lookahead sources are validated:
1. Feature revisions (accounting data vintage via `feature_available_date`)
2. Prediction leakage (walk-forward integrity)
3. Earnings announcement timing (signal_date < entry_date)


In [ ]:
# ============================================================
# Point-in-Time Analysis — Setup and Validation
# ============================================================
import importlib
import src.pit_utils as pit_utils
importlib.reload(pit_utils)
from src.pit_utils import (
    build_pit_feature_panel,
    validate_pit,
    build_pit_prediction_panel,
    compute_pit_signal_decay,
    flag_earnings_pit_violations,
)
print("PIT utilities loaded.")


In [ ]:
# ============================================================
# Point-in-Time — Feature Panel Audit
# ============================================================
# feature_panel_df contains 'feature_available_date' as vintage column.
_pit_cols_present = [c for c in [
    "feature_available_date", "report_date", "rdq", "datadate", "vintage_date"
] if c in feature_panel_df.columns]
_vintage_col = _pit_cols_present[0] if _pit_cols_present else None

if _vintage_col:
    print(f"[PIT] Vintage column found: '{_vintage_col}'")
    _pit_validation = validate_pit(
        feature_panel_df=feature_panel_df,
        as_of_col="date",
        vintage_col=_vintage_col,
        ticker_col="ticker",
        raise_on_violation=False,
    )
    print(_pit_validation["summary"])
    if not _pit_validation["clean"]:
        print(f"\n[PIT] WARNING: {_pit_validation['n_violations']} lookahead violations detected!")
        print(f"[PIT] Violation rate: {_pit_validation['violation_rate']:.2%}")
        print("\nSample violations:")
        print(_pit_validation["violations_df"].head(10).to_string(index=False))
    else:
        print("[PIT] Feature panel is CLEAN — no lookahead violations found.")
else:
    print("[PIT] WARNING: No vintage column found in feature_panel_df.")
    print("[PIT] Documenting assumption: features are implicitly PIT-safe by construction.")
    _pit_validation = {"clean": None, "n_violations": None, "summary": "No vintage col available"}

print("\n[PIT] Feature panel audit complete.")


In [ ]:
# ============================================================
# Point-in-Time — Build PIT-Safe Feature Panel
# ============================================================
if _vintage_col:
    feature_panel_pit = build_pit_feature_panel(
        feature_panel_df=feature_panel_df,
        as_of_col="date",
        vintage_col=_vintage_col,
        ticker_col="ticker",
        fill_method="ffill",
    )
    print(f"PIT-safe feature panel: {len(feature_panel_pit):,} rows")
    _lag_days = (feature_panel_pit["pit_vintage_date"]
                 .sub(feature_panel_pit["date"])
                 .dt.days)
    print("\nPIT vintage lag (days; negative = stale by that many days):")
    print(_lag_days.describe().round(1))
    print("\nSample PIT-safe panel (first 5 rows):")
    print(feature_panel_pit[["date", "ticker", "pit_vintage_date"]].head(5).to_string(index=False))
else:
    feature_panel_pit = feature_panel_df.copy()
    feature_panel_pit["pit_vintage_date"] = pd.to_datetime(feature_panel_pit["date"])
    print("[PIT] Using date column as proxy for vintage date.")
    print(f"PIT-safe feature panel (proxy): {len(feature_panel_pit):,} rows")


In [ ]:
# ============================================================
# Point-in-Time — Predictions Panel Integrity
# ============================================================
pit_predictions = build_pit_prediction_panel(
    predictions_df=predictions_df,
    feature_panel_pit=feature_panel_pit,
    signal_date_col="date",
    ticker_col="ticker",
)

print(f"PIT prediction panel: {len(pit_predictions):,} rows")
print()

if "pit_safe" in pit_predictions.columns:
    _n_safe   = pit_predictions["pit_safe"].sum()
    _n_unsafe = (~pit_predictions["pit_safe"]).sum()
    _total    = len(pit_predictions)
    print(f"  PIT-safe signals:    {_n_safe:,} / {_total:,} ({_n_safe/_total:.1%})")
    print(f"  PIT-unsafe signals:  {_n_unsafe:,} / {_total:,} ({_n_unsafe/_total:.1%})")
    if _n_unsafe > 0:
        print(f"\n[PIT] CRITICAL: {_n_unsafe} predictions use future data:")
        print(pit_predictions[~pit_predictions["pit_safe"]]
              [["date", "ticker", "pit_feature_lag"]].head(10).to_string(index=False))
    else:
        print("\n[PIT] All predictions are PIT-safe.")

if "pit_feature_lag" in pit_predictions.columns:
    _valid_lags = pit_predictions["pit_feature_lag"].dropna()
    if len(_valid_lags) > 0:
        print(f"\n  Avg feature staleness: {_valid_lags.mean():.1f} days")
        print(f"  P25/P50/P75 staleness: "
              f"{_valid_lags.quantile(0.25):.0f} / "
              f"{_valid_lags.quantile(0.50):.0f} / "
              f"{_valid_lags.quantile(0.75):.0f} days")

if "pit_safe" in pit_predictions.columns and pit_predictions["pit_safe"].notna().any():
    _valid = pit_predictions[pit_predictions["pit_safe"].notna()]
    assert _valid["pit_safe"].all(), "CRITICAL PIT VIOLATION: predictions use future feature data!"
    print("\nPIT prediction integrity assertion PASSED.")


In [ ]:
# ============================================================
# Point-in-Time — Signal Decay and Feature Staleness
# ============================================================
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle("Point-in-Time Analysis — Signal Integrity", fontsize=13, fontweight="bold")

# Panel 1: Feature staleness distribution
ax = axes[0]
if "pit_feature_lag" in pit_predictions.columns:
    _lags = pit_predictions["pit_feature_lag"].dropna()
    if len(_lags) > 0:
        ax.hist(_lags, bins=40, color="steelblue", edgecolor="white", alpha=0.85)
        ax.axvline(0, color="crimson", linewidth=1.5, linestyle="--",
                   label="No staleness (lag=0)")
        ax.axvline(_lags.mean(), color="darkorange", linewidth=1.5, linestyle="--",
                   label=f"Mean lag: {_lags.mean():.0f} days")
        ax.set_title("Feature Data Staleness Distribution")
        ax.set_xlabel("Days between feature vintage and signal date")
        ax.set_ylabel("Count")
        ax.legend(fontsize=9)
        _xlim = ax.get_xlim()
        if _xlim[0] < 0:
            ax.axvspan(_xlim[0], 0, alpha=0.1, color="crimson")
    else:
        ax.text(0.5, 0.5, "No lag data available", ha="center", va="center",
                transform=ax.transAxes)
else:
    ax.text(0.5, 0.5, "pit_feature_lag not available", ha="center", va="center",
            transform=ax.transAxes)

# Panel 2: Signal decay — prediction strength vs staleness quintile
ax = axes[1]
_has_lag  = "pit_feature_lag" in pit_predictions.columns
_has_pred = "prediction" in pit_predictions.columns
_enough   = _has_lag and pit_predictions["pit_feature_lag"].dropna().shape[0] > 4
if _has_lag and _has_pred and _enough:
    _decay = compute_pit_signal_decay(
        pit_panel=pit_predictions,
        signal_col="prediction",
        lag_col="pit_feature_lag",
        n_bins=5,
    )
    if not _decay.empty:
        ax.bar(range(len(_decay)), _decay["mean_prediction"],
               color="seagreen", edgecolor="white", alpha=0.8)
        _se = _decay["std_prediction"] / np.sqrt(_decay["n_signals"].clip(1))
        ax.errorbar(range(len(_decay)), _decay["mean_prediction"],
                    yerr=_se, fmt="none", color="black", capsize=4)
        ax.set_xticks(range(len(_decay)))
        ax.set_xticklabels(_decay["lag_bin"], rotation=30, ha="right", fontsize=8)
        ax.set_title("Prediction Strength by Feature Staleness Quintile")
        ax.set_xlabel("Feature lag quintile (fresher -> staler)")
        ax.set_ylabel("Mean LSTM prediction")
        ax.axhline(0, color="black", linewidth=0.5)
    else:
        ax.text(0.5, 0.5, "Insufficient data for decay analysis",
                ha="center", va="center", transform=ax.transAxes)
else:
    ax.text(0.5, 0.5, "Decay analysis requires pit_feature_lag + prediction columns",
            ha="center", va="center", transform=ax.transAxes)

plt.tight_layout()
_fig_dir = Path("..") / "docs" / "figures"
_fig_dir.mkdir(parents=True, exist_ok=True)
plt.savefig(str(_fig_dir / "pit_analysis.png"), dpi=120, bbox_inches="tight")
plt.show()
print("PIT analysis figure saved.")


In [ ]:
# ============================================================
# Point-in-Time — Summary Report
# ============================================================
_pit_summary = {
    "Feature panel vintage col": _vintage_col or "NOT FOUND (proxy used)",
    "Feature panel size": f"{len(feature_panel_pit):,} rows",
    "PIT violations in feature panel": (
        str(_pit_validation.get("n_violations", "N/A"))
        if _pit_validation.get("n_violations") is not None else "N/A"
    ),
    "Prediction panel PIT-safe rate": (
        f"{pit_predictions['pit_safe'].mean():.1%}"
        if "pit_safe" in pit_predictions.columns else "N/A"
    ),
    "Avg feature staleness (days)": (
        f"{pit_predictions['pit_feature_lag'].dropna().mean():.1f}"
        if "pit_feature_lag" in pit_predictions.columns
           and pit_predictions["pit_feature_lag"].dropna().shape[0] > 0
        else "N/A"
    ),
    "Earnings timing check (all valid)": (
        str(pit_trade_flags["pit_valid"].all())
        if pit_trade_flags is not None and "pit_valid" in pit_trade_flags.columns
        else "Not run"
    ),
}

print("=" * 56)
print("POINT-IN-TIME ANALYSIS — FINAL SUMMARY")
print("=" * 56)
for k, v in _pit_summary.items():
    print(f"  {k:<40}: {v}")
print("=" * 56)


In [ ]:
# ============================================================
# Integration Test — Trading Costs + PIT Combined
# All assertions below must pass for this implementation to be
# considered complete and correct.
# ============================================================
import pandas as pd
import numpy as np

print("Running integration tests...")
print()

# Test 1: Cost model produces valid marks on full trade_log
_exits = marked_trades[marked_trades["action"] == "exit"].copy()
assert len(_exits) > 0, "No exit trades to test"
assert "cost_total" in _exits.columns, "cost_total column missing"
assert "net_pnl" in _exits.columns, "net_pnl column missing"
assert (_exits["cost_total"].dropna() >= 0).all(), "Negative costs found"
print(f"[PASS] Test 1: Cost model — {len(_exits)} trades marked, all costs >= 0")

# Test 2: Net PnL is always <= gross PnL
_both = _exits[_exits["net_pnl"].notna() & _exits["realized_pnl"].notna()]
assert (_both["net_pnl"] <= _both["realized_pnl"] + 1e-8).all(), \
    "Net PnL exceeds gross PnL (impossible)"
print(f"[PASS] Test 2: Net PnL <= Gross PnL for all {len(_both)} exit trades")

# Test 3: Total cost drag is economically plausible (report, not hard assert)
_avg_cost = _exits["cost_total"].mean()
_avg_pnl  = _exits["realized_pnl"].abs().mean()
_drag_pct = _avg_cost / (_avg_pnl + 1e-8)
print(f"[INFO] Test 3: Avg cost ${_avg_cost:.2f}  |  avg |gross PnL| ${_avg_pnl:.2f}  |  ratio {_drag_pct:.2%}")

# Test 4: PIT predictions have non-negative lag (no future data)
if "pit_feature_lag" in pit_predictions.columns:
    _valid_lags = pit_predictions["pit_feature_lag"].dropna()
    if len(_valid_lags) > 0:
        _neg_lag = (_valid_lags < 0).sum()
        assert _neg_lag == 0, f"PIT FAIL: {_neg_lag} predictions use future features"
        print(f"[PASS] Test 4: PIT — no predictions use future feature data")
    else:
        print(f"[SKIP] Test 4: no non-null pit_feature_lag values")
else:
    print(f"[SKIP] Test 4: pit_feature_lag not available")

# Test 5: Signal dates precede entry dates for all trades
_entries = trade_log[trade_log["action"] == "enter"].copy()
if "signal_date" in _entries.columns:
    _entries["signal_date"] = pd.to_datetime(_entries["signal_date"], errors="coerce")
    _entries["date"]        = pd.to_datetime(_entries["date"], errors="coerce")
    _both_d = _entries["signal_date"].notna() & _entries["date"].notna()
    if _both_d.any():
        _bad = (_entries.loc[_both_d, "signal_date"]
                >= _entries.loc[_both_d, "date"]).sum()
        assert _bad == 0, f"PIT FAIL: {_bad} trades where signal_date >= entry_date"
        print(f"[PASS] Test 5: PIT — all signal_dates strictly precede entry_dates")
    else:
        print(f"[SKIP] Test 5: no rows with both dates non-null")
else:
    print(f"[SKIP] Test 5: signal_date column not in trade_log")

# Test 6: Profit factor is well-defined
_net_wins   = _exits[_exits["net_pnl"] > 0]["net_pnl"].sum()
_net_losses = _exits[_exits["net_pnl"] < 0]["net_pnl"].sum()
assert _net_losses != 0, "All trades profitable — profit factor undefined; check cost model"
_pf = _net_wins / abs(_net_losses)
print(f"[PASS] Test 6: Net Profit Factor = {_pf:.3f}  (well-defined)")
print()

print("=" * 50)
print("ALL INTEGRATION TESTS PASSED")
print("=" * 50)
print(f"\nFinal summary:")
print(f"  Gross total PnL:      ${_exits['realized_pnl'].sum():>12,.2f}")
print(f"  Total costs:           ${_exits['cost_total'].sum():>12,.2f}")
print(f"  Net total PnL:         ${_exits['net_pnl'].sum():>12,.2f}")
print(f"  Cost drag:             ${(_exits['realized_pnl'].sum() - _exits['net_pnl'].sum()):>12,.2f}")
print(f"  Gross win rate:        {(_exits['realized_pnl'] > 0).mean():.1%}")
print(f"  Net win rate:          {(_exits['net_pnl'] > 0).mean():.1%}")
print(f"  Net Profit Factor:     {_pf:.3f}")
